# 01 · Analysis and processing of the radiomics features

## The pipeline so far

| step | script / notebook | what it produces |
|---|---|---|
| 1. preprocessing | `src/mass_detect/scripts/lidc_preprocessing.py` (LIDC, needs `.venv-lidc`) | nodule volumes + consensus masks in `data/nodules/` |
| 1. preprocessing | `src/mass_detect/scripts/prepare_lndb_nodules.py` (LNDb) | 64 mm cubes + automatic masks in `data/nodules/<split>/` |
| 2. extraction | `src/mass_detect/scripts/extract_features_pyrad.py` | **`data/radiomics_features.csv`** — one row per nodule, one column per PyRadiomics feature |
| 3. analysis | **this notebook** | `data/radiomics_features_processed.csv` (cleaned, filtered) |
| 4. modelling | `notebooks/02_train_model_and_predict.ipynb` | cross-validated metrics + `data/test_predictions.csv` |

Everything is configured in the first code cell of each notebook: change
paths and thresholds there, not in the cells below.

---

**Context you should know before reviewing this notebook.** A teammate
already built a full model on 30 handcrafted features (see
`notebooks/6out_summary.txt`). Her key results, which this work is
compared against:

* Her 30-feature ordinal model, validated on the official LNDb folds:
  balanced accuracy 0.411, macro F1 0.429, QWK 0.533, MAE 0.654.
* **Her radiomics experiment failed**: with an automatic segmentation of
  the nodule (the only option for the test set, which has centroids
  only), PyRadiomics features (386 of them, robustness-filtered) did
  *not* beat the 30 handcrafted features (composite 0.561 vs 0.651).
  Her diagnosis: features from an automatic mask agree with features
  from a radiologist's mask only at ICC ~ 0.73 — the segmentation, not
  the features, is the bottleneck.
* The radiologist-vs-radiologist agreement (the noise ceiling of the
  labels) is QWK ~ 0.527 on LNDb.

This notebook is the parallel, *radiomics-intensive* track: it takes the
same idea much further (386 features, 4 image types, all feature
classes), cleans and filters them transparently, and lets the modelling
notebook measure — with the same validation protocol — whether the more
intensive use of PyRadiomics pays off.


## What this notebook does

1. **Loads** `data/radiomics_features.csv` (produced by
   `extract_features_pyrad.py`) together with the LNDb manifests (quality
   control columns such as how much of each cube falls outside the CT) and
   the malignancy **labels** of both datasets.
2. **Explores** the features: how many there are per image type and feature
   class, how many are missing, which ones are constant, how they are
   distributed, and how they correlate with the malignancy rating.
3. **Processes** them into a modelling-ready table:
   * separates the `diagnostics_*` columns (they are metadata, not features);
   * drops constant features and features that are missing in more than a
     configurable share of training rows;
   * removes redundancy with a **correlation filter** (greedy removal of
     feature pairs whose absolute Spearman correlation exceeds a threshold).
4. **Saves** the processed table (`data/radiomics_features_processed.csv`),
   the list of kept features and a report of everything that was removed
   and why.

## Why the correlation filter is *unsupervised*

Choosing between two correlated features *with the label* (e.g. keeping the
one more correlated with malignancy) is legitimate — but only inside the
cross-validated model selection, where the choice is refitted on every
training fold. Doing it once, on all training rows, and baking the result
into a static CSV would leak label information into every later experiment.
The filter here therefore uses no label: between two highly correlated
features it keeps the one with fewer missing values (ties broken
alphabetically, deterministically). Label-based selection happens in
notebook 02, inside each cross-validation fold (`SELECT_K_BEST` there).

## Where the labels come from

* **LNDb**: the per-reading malignancy ratings live in `chars_trainNodules.csv`
  (not in this repository checkout). If you place that file where configured,
  this notebook rebuilds the full per-finding label exactly as the LNDb
  scripts do (positional join with `trainNodules.csv`, then averaging per
  finding — *never* joining on `RadID`, which means different things in the
  two files; see `notebooks/6out_summary.txt` for why). Without that file it
  falls back to `notebooks/lndb_processed.csv`, a per-finding table that
  covers 659 of the 1219 training findings. **This is the main limitation of
  the current run**: about 100 rated nodules cannot be used for training
  until `chars_trainNodules.csv` is added.
* **LIDC**: `data/nodules/meta_info_lidc.csv` (written by a *complete*
  run of the LIDC preprocessing) or, until then,
  `data/nodules/lidc_labels.csv` (written by
  `src/mass_detect/scripts/lidc_labels.py`, which recomputes the same
  labels with pylidc).


In [ ]:
# ============================================================
# CONFIGURATION — change paths and thresholds here, nowhere else
# ============================================================
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# ---------- path handling ----------
# Notebooks are executed either from the repository root (jupyter,
# scripts) or from notebooks/ (nbconvert executes a notebook with
# its own directory as the working directory). Every configured
# path is therefore resolved against the repository root, so both
# ways of running work.
def _repo_root():
    here = Path.cwd()
    return here if (here / "pyproject.toml").exists() else here.parent

REPO_ROOT = _repo_root()

def _resolve(path):
    """Relative paths are resolved against the repository root."""
    path = Path(path)
    return path if path.is_absolute() else REPO_ROOT / path

# ---------- input: raw features (from extract_features_pyrad.py) ----------
RAW_FEATURES_CSV = _resolve("data/radiomics_features.csv")

# ---------- LNDb manifests (QC columns per cube) ----------
# split -> manifest path; a missing manifest is skipped, not an error
LNDB_MANIFESTS = {
    "train": _resolve("data/nodules/train/lndb_manifest.csv"),
    "test": _resolve("data/nodules/test/lndb_manifest.csv"),
}

# ---------- LNDb labels ----------
# Full label source (preferred): per-reading ratings (chars_trainNodules.csv)
# positionally joined with the readings (trainNodules.csv), then averaged
# per finding (trainNodules_gt.csv). chars_trainNodules.csv is NOT in this
# checkout — leave the path as is and the notebook falls back.
CHARS_CSV = _resolve("data/LNDb/data/chars_trainNodules.csv")
READINGS_CSV = _resolve("tmp/trainNodules.csv")
FINDINGS_GT_CSV = _resolve("tmp/trainNodules_gt.csv")
# Fallback label source (what this checkout ships): per-finding table
# built in notebooks/explorar_lndb.ipynb, covering 659 findings.
LNDB_PROCESSED_CSV = _resolve("notebooks/lndb_processed.csv")

# ---------- LIDC labels ----------
# meta_info_lidc.csv is written by a complete run of lidc_preprocessing.py;
# lidc_labels.csv is the equivalent produced by scripts/lidc_labels.py.
LIDC_META_CSV = _resolve("data/nodules/meta_info_lidc.csv")
LIDC_LABELS_CSV = _resolve("data/nodules/lidc_labels.csv")
# Which column of the LIDC label file to use as the label.
# lidc_labels.csv offers: "malignancy_median_high" (reproduces what
# the preprocessing stores), "malignancy_mean" (a softer label),
# "malignancy_min", "malignancy_max". meta_info_lidc.csv (a
# complete preprocessing run) only has "malignancy".
LIDC_LABEL = "malignancy_median_high"

# ---------- processing thresholds ----------
# Drop features missing in more than this share of the LNDb training rows.
# Beyond it, median imputation would replace most of the values and the
# "feature" would mostly describe the imputation.
MISSINGNESS_THRESHOLD = 0.20
# Greedy removal of feature pairs with |Spearman rho| above this.
# 0.90 is the usual cutoff in radiomics (Traverso et al., 2018); it removes
# redundancy while keeping genuinely different signals.
CORR_THRESHOLD = 0.90

# ---------- output ----------
PROCESSED_CSV = _resolve("data/radiomics_features_processed.csv")
KEPT_FEATURES_CSV = _resolve("data/radiomics_kept_features.csv")
PROCESSING_REPORT_CSV = _resolve("data/radiomics_processing_report.csv")

# ---------- display ----------
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False})
sns.set_theme(style="whitegrid", font_scale=0.9)
print("configuration loaded")

## 1 · Load the raw features

The extraction script writes one row per nodule and one column per
PyRadiomics feature, named `<imageType>_<featureClass>_<featureName>`
(e.g. `original_firstorder_Mean`,
`log-sigma-2-0-mm-3D_glcm_Correlation`). The `diagnostics_*` columns are
metadata about the images (spacing, dimensions, the software version),
not features — they are kept aside for the quality-control section below.

Rows with `status = error` are nodules whose extraction failed (for LNDb:
findings where the automatic segmentation found no nodule-like structure,
about 6% of the training findings). Their feature columns are empty;
they stay in the table so the modelling notebook can impute them and keep
the "segmentation failed" information as a signal of its own.

In [ ]:
raw = pd.read_csv(RAW_FEATURES_CSV)
print(f"rows: {len(raw)}")

# Separate metadata, PyRadiomics features and diagnostics.
meta_cols = ["nodule_id", "dataset", "split", "image_path", "mask_path",
             "status", "error"]
feature_cols = [c for c in raw.columns if c not in meta_cols
                and not c.startswith("diagnostics_")]
diagnostic_cols = [c for c in raw.columns if c.startswith("diagnostics_")]

print(f"feature columns: {len(feature_cols)} | diagnostic columns: {len(diagnostic_cols)}")

# Inventory: how many features per image type and per feature class.
# PyRadiomics names features <imageType>_<featureClass>_<name>, and the
# image types we enabled contain no underscore ("original",
# "log-sigma-1-0-mm-3D", ...), so the split below is unambiguous.
def _image_type(name):
    return name.split("_")[0]

def _feature_class(name):
    return name.split("_")[1]

inventory = pd.DataFrame({"image_type": [_image_type(c) for c in feature_cols],
                          "feature_class": [_feature_class(c) for c in feature_cols]})
print("\nfeatures per image type:")
print(inventory["image_type"].value_counts().to_string())
print("\nfeatures per image type x feature class:")
print(pd.crosstab(inventory["image_type"], inventory["feature_class"]).to_string())

# Overview of the rows.
print("\nrows per dataset / split:")
print(raw.groupby(["dataset", "split"]).size().to_string())
print("\nextraction status per dataset:")
print(pd.crosstab(raw["dataset"], raw["status"]).to_string())
raw.head(3)

## 2 · Quality control

Two things are worth checking before any modelling:

1. **The resampling worked.** `Params.yml` asks PyRadiomics to resample
   every volume to 1 mm isotropic voxels. PyRadiomics reports the spacing
   of the volume as it was read (`diagnostics_*_original_Spacing`, the
   native, possibly anisotropic spacing) and of the volume after
   resampling (`diagnostics_*_interpolated_Spacing`). The latter must be
   `(1.0, 1.0, 1.0)` for every nodule, whatever the source — only then
   are features from the two datasets comparable (texture features depend
   on voxel size: Shafiq-ul-Hassan et al., 2017).
2. **The LNDb cubes are healthy.** The manifest of each split records, per
   finding, the fraction of cube voxels that fall outside the CT (cubes
   near the chest wall), whether the automatic segmentation found anything,
   and how many voxels the mask has. A large share of unsegmented or tiny
   masks would warn us that the features of those rows are mostly imputed.


In [ ]:
def _parse_tuple_column(series):
    """Parse a diagnostics column of stringified tuples, e.g. '(1.0, 1.0, 1.0)'."""
    import ast
    out = []
    for value in series:
        try:
            out.append(tuple(float(v) for v in ast.literal_eval(str(value))))
        except (ValueError, SyntaxError):
            out.append(None)
    return out

# --- 2.1 did the resampling to 1 mm isotropic work everywhere? ---
interp_spacing = raw.get("diagnostics_Image-interpolated_Spacing")
if interp_spacing is not None:
    spacings = pd.Series(_parse_tuple_column(interp_spacing.dropna()))
    bad = spacings[spacings != (1.0, 1.0, 1.0)]
    print(f"volumes checked: {len(spacings)} | "
          f"not resampled to (1, 1, 1): {len(bad)}")
    if len(bad):
        print(bad.head().to_string())
    native = pd.Series(_parse_tuple_column(
        raw["diagnostics_Image-original_Spacing"].dropna()))
    print("native spacings of the source volumes (before resampling), "
          "most common:")
    print(native.value_counts().head(5).to_string())
else:
    print("[note] no spacing diagnostics found in the features file")

# --- 2.2 LNDb manifest QC ---
# The manifest also carries the columns the submission needs
# (LNDbID, FindingID, x, y, z), so they are merged too.
manifest_frames = []
for split, path in LNDB_MANIFESTS.items():
    if not Path(path).exists():
        print(f"[note] manifest for split '{split}' not found at {path} (skipped)")
        continue
    m = pd.read_csv(path)
    m["split"] = split
    manifest_frames.append(m)
lndb_manifest = pd.concat(manifest_frames, ignore_index=True) if manifest_frames else None

if lndb_manifest is not None:
    qc_cols = [c for c in lndb_manifest.columns
               if c not in ("status", "error")]   # keep the extraction's own
    before = len(raw)
    raw = raw.merge(lndb_manifest[qc_cols], on=["nodule_id", "split"], how="left")
    assert len(raw) == before, "manifest merge changed the number of rows"
    print("\nLNDb QC (per split):")
    qc_summary = (raw[raw.dataset == "LNDb"]
                  .groupby("split")
                  .agg(rows=("nodule_id", "size"),
                       segmented_share=("segmented", "mean"),
                       median_outside_fraction=("outside_fraction", "median"),
                       max_outside_fraction=("outside_fraction", "max"),
                       median_mask_voxels=("n_voxels", "median"),
                       extraction_errors=("status", lambda s: (s == "error").mean())))
    print(qc_summary.round(3).to_string())

    # Cubes with more than 5% of voxels outside the CT (kept, but flagged —
    # their edge voxels are air padding).
    edge = raw[raw["outside_fraction"] > 0.05]
    print(f"\ncubes with >5% outside voxels: {len(edge)} "
          f"({len(edge) / max((raw.dataset == 'LNDb').sum(), 1):.1%} of LNDb rows)")

# Missingness: a failed extraction leaves every feature empty.
missing_share = raw[feature_cols].isna().mean()
print(f"\nfeatures with ANY missing value: {(missing_share > 0).sum()} of {len(feature_cols)}")
print("most missing features:")
print(missing_share.sort_values(ascending=False).head(10).round(3).to_string())

## 3 · Labels

The target is the **mean malignancy rating of the radiologists, rounded
half-up to the integer 1-5** (2.5 becomes 3), exactly as the competition
and the teammate's model define it. Two derived views are also kept — a
ternary one (benign / indeterminate / malignant) and a binary one — so
the modelling notebook can evaluate on the clinically easier question
without rebuilding anything.

Nothing is dropped for having an indeterminate rating: the competition
asks for a value for every candidate, and silently excluding them makes
the problem look easier than it is.

One column from the manifest matters for modelling: `nodule`, 1 when the
merged LNDb finding is a nodule. Findings that the merged record calls
non-nodules are **not** used for training malignancy, even when one
radiologist rated them: the rating would describe a nodule that the other
radiologists did not see, i.e. a contradictory label (the same reasoning
as in the teammate's Part 3).

In [ ]:
def lndb_labels():
    """Per-finding LNDb malignancy labels, as a table indexed by nodule_id.

    Preferred source: chars_trainNodules.csv (per-reading ratings), joined
    with trainNodules.csv BY ROW POSITION (the two files list the same
    readings in the same order; joining on RadID would mix up the ratings,
    because RadID is CT-local in one file and global in the other). The
    ratings are then expanded through the radiologist lists of
    trainNodules_gt.csv and averaged per finding.

    Fallback (when chars_trainNodules.csv is absent): the per-finding
    table notebooks/lndb_processed.csv, which covers fewer findings.
    """
    chars_path = Path(CHARS_CSV)
    if chars_path.exists():
        readings = pd.read_csv(READINGS_CSV)
        chars = pd.read_csv(chars_path)
        assert len(readings) == len(chars), "readings and ratings files differ in length"
        assert (readings[["LNDbID", "FindingID"]].values
                == chars[["LNDbID", "FindingID"]].values).all(), \
            "readings and ratings files are not row-aligned"
        # 0 means "not filled in" by the radiologist, not a benign nodule.
        readings["malignancy"] = chars["malignancy"].replace(0, np.nan).to_numpy()

        findings = pd.read_csv(FINDINGS_GT_CSV)
        rows = []
        for finding in findings.itertuples():
            rad_ids = str(finding.RadID).split(",")          # CT-local radiologist ids
            reading_ids = str(finding.RadFindingID).split(",")  # their finding numbers
            ratings = []
            for rad_id, reading_id in zip(rad_ids, reading_ids):
                hit = readings[(readings.LNDbID == finding.LNDbID)
                               & (readings.RadID == int(rad_id))
                               & (readings.FindingID == int(reading_id))]
                if len(hit) == 1:
                    ratings.append(hit["malignancy"].iloc[0])
            ratings = [r for r in ratings if pd.notna(r)]
            rows.append({"LNDbID": int(finding.LNDbID),
                         "FindingID": int(finding.FindingID),
                         "mal_mean": np.mean(ratings) if ratings else np.nan,
                         "mal_min": min(ratings) if ratings else np.nan,
                         "mal_max": max(ratings) if ratings else np.nan,
                         "n_rated": len(ratings)})
        labels = pd.DataFrame(rows)
        labels["label_source"] = "chars_trainNodules.csv (full)"
        print(f"LNDb labels from {CHARS_CSV}: {labels['mal_mean'].notna().sum()} rated findings")
    else:
        print(f"[note] {CHARS_CSV} not found — falling back to {LNDB_PROCESSED_CSV} "
              "(covers fewer findings)")
        table = pd.read_csv(LNDB_PROCESSED_CSV)
        labels = table.rename(columns={"scan_id": "LNDbID", "finding_id": "FindingID",
                                       "malignancy": "mal_mean"})[
            ["LNDbID", "FindingID", "mal_mean"]].copy()
        labels["mal_min"] = labels["mal_max"] = np.nan
        labels["n_rated"] = np.nan
        labels["label_source"] = "lndb_processed.csv (partial)"
        print(f"LNDb labels from fallback: {labels['mal_mean'].notna().sum()} rated findings")

    labels["nodule_id"] = [f"LNDb-{int(ct):04d}_nodule{int(f):03d}"
                           for ct, f in zip(labels.LNDbID, labels.FindingID)]
    # The modelling code expects the label in a column called
    # "malignancy" (the name the LIDC label tables use); keep
    # "mal_mean" as well because it carries the extra meaning
    # "mean of the radiologists' ratings".
    labels["malignancy"] = labels["mal_mean"]
    return labels


def lidc_labels():
    """Per-nodule LIDC malignancy labels, indexed by nodule_id."""
    meta = Path(LIDC_META_CSV)
    if meta.exists():
        table = pd.read_csv(meta)
        table["nodule_id"] = table["patient_id"] + "_nodule" + table["nodule_idx"].astype(str)
        label_col = "malignancy"          # column name in meta_info_lidc.csv
        print(f"LIDC labels from {LIDC_META_CSV}: {len(table)} nodules")
    else:
        table = pd.read_csv(LIDC_LABELS_CSV)
        label_col = LIDC_LABEL            # e.g. "median_high" or "mean"
        print(f"LIDC labels from {LIDC_LABELS_CSV} (column '{label_col}'): {len(table)} nodules")
    table["malignancy"] = table[label_col]
    return table[["nodule_id", "malignancy"]]


# --- join the labels onto the feature table ---
labels = pd.concat([lndb_labels(), lidc_labels()], ignore_index=True)
data = raw.merge(labels, on="nodule_id", how="left")

# Derived label views (see the markdown cell above).
rated = data["malignancy"].notna()
data["rating"] = np.floor(data["malignancy"] + 0.5).where(rated)   # round half-up -> 1..5
data["rating_ternary"] = pd.cut(data["malignancy"], bins=[0, 2.5, 3.5, 5],
                                labels=["benign", "indeterminate", "malignant"]).where(rated)
data["rating_binary"] = data["rating_ternary"].map(
    {"benign": 0, "malignant": 1}).where(rated)

print("\nlabelled rows per dataset (the model trains on these):")
print(data[rated].groupby("dataset").size().to_string())
print("\nclass distribution of the rounded rating (LNDb):")
print(data.loc[rated & (data.dataset == "LNDb"), "rating"].value_counts()
      .sort_index().to_string())
print("\nclass distribution of the rounded rating (LIDC):")
print(data.loc[rated & (data.dataset == "LIDC"), "rating"].value_counts()
      .sort_index().to_string())
print(f"\nunlabelled rows (test candidates have no label by design): "
      f"{(~rated).sum()} of {len(data)}")

## 4 · Exploratory analysis of the features

* **Missingness**: which features are undefined for many nodules. For LNDb
  essentially all missingness comes from whole *rows* (failed
  segmentations), but the check is generic — some features can also be
  undefined for degenerate regions (e.g. texture features of a region
  with too few distinct gray levels).
* **Constant features**: carry no information and can destabilise scalers.
* **Distributions**: a sanity look at a few well-known features.
* **Signal**: Spearman correlation of every feature with the rating, on
  the LNDb training rows (the target domain of the competition). Spearman
  is used because the relations are monotone but not linear, and it is
  robust to outliers.

In [ ]:
train_lndb = (data.dataset == "LNDb") & (data.split == "train")

# --- 4.1 missingness and constant features ---
missing_on_train = data.loc[train_lndb, feature_cols].isna().mean()
print(f"features missing in > {MISSINGNESS_THRESHOLD:.0%} of LNDb training rows: "
      f"{(missing_on_train > MISSINGNESS_THRESHOLD).sum()}")
constant = [c for c in feature_cols
            if data.loc[train_lndb, c].nunique(dropna=True) <= 1]
print(f"constant features on LNDb training rows: {len(constant)} {constant[:5]}")

# --- 4.2 example distributions (before any processing) ---
examples = ["original_firstorder_Mean",
            "original_shape_Maximum3DDiameter",
            "original_firstorder_Energy"]
examples = [c for c in examples if c in feature_cols]
fig, axes = plt.subplots(1, len(examples), figsize=(4.2 * len(examples), 3))
for ax, col in zip(np.atleast_1d(axes), examples):
    data.loc[train_lndb, col].dropna().hist(ax=ax, bins=40, color="C0")
    ax.set_title(col.replace("_", " "), fontsize=9)
plt.tight_layout(); plt.show()

# --- 4.3 signal: Spearman correlation with the rating ---
corr_with_rating = (data.loc[train_lndb, feature_cols]
                    .corrwith(data.loc[train_lndb, "rating"], method="spearman")
                    .sort_values())
print(f"computed for {corr_with_rating.notna().sum()} features on "
      f"{int(train_lndb.sum())} LNDb training rows")
fig, ax = plt.subplots(figsize=(6, 7))
corr_with_rating.tail(20).plot.barh(ax=ax, color="C0")
ax.set_xlabel("Spearman rho with the malignancy rating")
ax.set_title("20 features most (anti-)correlated with malignancy, LNDb train")
plt.tight_layout(); plt.show()

# --- 4.4 the strongest feature, per class ---
top = corr_with_rating.abs().idxmax()
print(f"strongest feature: {top} (rho = {corr_with_rating[top]:.2f})")
fig, ax = plt.subplots(figsize=(6, 3.5))
sns.boxplot(data=data[train_lndb], x="rating", y=top,
            legend=False, ax=ax)
ax.set_title(f"{top} by rating class (LNDb train)")
plt.tight_layout(); plt.show()

## 5 · Processing: from raw to modelling-ready features

The steps, in order, and why:

1. **Drop `diagnostics_*`** — they describe the images and the software,
   not the nodule (and would leak e.g. the slice thickness into the model).
2. **Drop constant features** — no information, and they break scalers.
3. **Drop features missing in more than `MISSINGNESS_THRESHOLD` of the LNDb
   training rows** — a feature that is mostly imputed describes the
   imputation, not the nodule.
4. **Correlation filter** (the main step): compute the Spearman correlation
   matrix once, on the LNDb training rows, then greedily remove the most
   correlated pair until no pair exceeds `CORR_THRESHOLD`. Between the two
   features of a pair, the one with **fewer missing values** survives
   (ties: alphabetical order) — a deterministic rule that uses no label,
   see the notebook introduction for why.

The filter is **fitted on the LNDb training rows only and applied to every
row** (LIDC and test rows included). It is an unsupervised transformation,
so applying it to other rows leaks nothing; fitting it on the target
domain (LNDb) keeps the feature set relevant to the competition's test
set, which is LNDb.

The report of what was removed, and why, is saved to
`data/radiomics_processing_report.csv` so every decision can be audited.

In [ ]:
# --- step 0: which rows the filter is fitted on ---
fit_rows = train_lndb & data["rating"].notna()
print(f"correlation filter fitted on {int(fit_rows.sum())} LNDb training rows")

# --- steps 1-3: diagnostics, constant, high-missingness ---
report = []          # one row per removed feature: name, reason, details
def drop(columns, reason, details=None):
    for col in columns:
        report.append({"feature": col, "reason": reason,
                       "details": details if details is not None else ""})
    return [c for c in feature_cols if c not in set(columns)]

candidates = drop(diagnostic_cols, "diagnostic metadata (not a feature)")
candidates = drop(constant, "constant on the LNDb training rows")

high_missing = missing_on_train[missing_on_train > MISSINGNESS_THRESHOLD].index.tolist()
candidates = drop(high_missing, f"missing in > {MISSINGNESS_THRESHOLD:.0%} of LNDb training rows",
                  details=missing_on_train[high_missing].round(3).to_dict())
print(f"after steps 1-3: {len(candidates)} candidate features")

# --- step 4: greedy correlation filter (label-free) ---
corr_matrix = data.loc[fit_rows, candidates].corr(method="spearman").abs()
missing_on_fit = data.loc[fit_rows, candidates].isna().mean()

corr_values = corr_matrix.to_numpy()
np.fill_diagonal(corr_values, 0.0)
kept_mask = np.ones(len(candidates), dtype=bool)
removed_by_corr = []
while True:
    alive = kept_mask[None, :] & kept_mask[:, None]
    masked = np.where(alive, corr_values, 0.0)
    i, j = np.unravel_index(np.argmax(masked), masked.shape)
    if masked[i, j] <= CORR_THRESHOLD:
        break
    # remove the more missing one; ties: later alphabetical order
    name_i, name_j = candidates[i], candidates[j]
    if (missing_on_fit[name_i], name_i) > (missing_on_fit[name_j], name_j):
        loser, winner = i, j
    else:
        loser, winner = j, i
    removed_by_corr.append({"feature": candidates[loser],
                            "removed_because_of": candidates[winner],
                            "spearman_rho": round(float(masked[i, j]), 3)})
    kept_mask[loser] = False

kept_features = [c for c, keep in zip(candidates, kept_mask) if keep]
for entry in removed_by_corr:
    report.append({"feature": entry["feature"],
                   "reason": f"|Spearman| > {CORR_THRESHOLD} with another feature",
                   "details": f"rho={entry['spearman_rho']} with {entry['removed_because_of']}"})

print(f"\ncorrelation filter removed {len(removed_by_corr)} features "
      f"(pairs above {CORR_THRESHOLD})")
print(f"kept: {len(kept_features)} features "
      f"(from {len(feature_cols)} raw columns)")

# What survives, per image type and class — a quick check that no whole
# image type or class was wiped out.
kept_inventory = pd.DataFrame({"image_type": [_image_type(c) for c in kept_features],
                               "feature_class": [_feature_class(c) for c in kept_features]})
print("\nkept features per image type x feature class:")
print(pd.crosstab(kept_inventory["image_type"], kept_inventory["feature_class"]).to_string())

# The strongest correlated groups that were collapsed, for the report.
removed_df = pd.DataFrame(removed_by_corr).sort_values("spearman_rho", ascending=False)
print("\nmost correlated pairs that were collapsed (winner kept, loser removed):")
print(removed_df.head(10).to_string(index=False))

In [ ]:
# --- assemble and save the processed table ---
# Metadata first (de-duplicated, order stable), then the labels, then the
# kept features.
keep_columns = list(dict.fromkeys(
    meta_cols + (qc_cols if lndb_manifest is not None else [])))
keep_columns = [c for c in keep_columns if c in data.columns]
processed = data[keep_columns + ["malignancy", "mal_mean", "mal_min", "mal_max",
                                 "n_rated", "label_source", "rating",
                                 "rating_ternary", "rating_binary"]
                 + kept_features].copy()

processed.to_csv(PROCESSED_CSV, index=False)
pd.DataFrame({"feature": kept_features}).to_csv(KEPT_FEATURES_CSV, index=False)
pd.DataFrame(report).to_csv(PROCESSING_REPORT_CSV, index=False)

print(f"saved {PROCESSED_CSV}: {len(processed)} rows x {len(processed.columns)} columns "
      f"({len(kept_features)} features)")
print(f"saved {KEPT_FEATURES_CSV} and {PROCESSING_REPORT_CSV} "
      f"({len(report)} removal records)")

# Sanity check: every feature column exists and is numeric.
assert all(np.issubdtype(processed[c].dtype, np.number) for c in kept_features)
processed.head(3)

## 6 · Limitations and things to check when you re-run this

* **Label coverage (LNDb).** Without `chars_trainNodules.csv` the labels
  come from `lndb_processed.csv` and cover only ~659 of the 1219 training
  findings (of which ~610 are rated nodules). The model trained in notebook
  02 therefore sees fewer nodules than the teammate's run (765). Add
  `chars_trainNodules.csv` at the configured path and re-run this notebook
  to unlock the full label set — no other change is needed.
* **Label coverage (LIDC).** The LIDC nodules extracted so far (1986 of
  2663) are the ones the interrupted preprocessing run produced. Re-running
  `lidc_preprocessing.py` on the whole dataset adds the rest; this
  notebook then picks them up automatically (the extraction script is
  resumable).
* **The two datasets are not cut the same way.** LIDC volumes are the
  nodule's bounding box plus a small padding at the native spacing (then
  resampled to 1 mm by PyRadiomics); LNDb cubes are fixed 64 mm cubes at
  1 mm. Features computed *on the region of interest* are equivalent
  either way (PyRadiomics crops to the ROI), but the Laplacian-of-Gaussian
  features are computed on the cropped image and can differ slightly at
  the ROI border. This is a known, accepted caveat of mixing the two
  datasets; notebook 02 validates on LNDb only, so any damage from it is
  measured, not assumed.
* **Failed segmentations** (LNDb rows with `status = error` or
  `segmented = False`) keep their row with empty features; the modelling
  notebook imputes them and adds the failure itself as a feature.